In [1]:
import tensorflow as tf
from datetime import date

c:\2022-Anaconda3\envs\tensorflow\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\2022-Anaconda3\envs\tensorflow\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\2022-Anaconda3\envs\tensorflow\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\2022-Anaconda3\envs\tensorflow\lib\site-packages\tensorflow\python\framework

## Application financière

Dans ce notebook, nous allons présenter une méthode de prédiction de prix de valeurs mobilières.

Ce notebook a été construit à partir de la source suivante : [*Practical Time Series Analysis, Prediction with Statistics & Machine Learning*, Aileen Nielsen, 2020](https://www.oreilly.com/library/view/practical-time-series/9781492041641/).

Avant de commencer la présentation de la méthode, nous allons charger nos données.

### Les données

Dans ce notebook, nous allons toujours utiliser nos données de valeurs mobilières.

In [2]:
%run -i "../BaseLoader.ipynb"

%run -i "../BaseClasses.ipynb"

%run -i "StrategiesModel.ipynb"

plt.rcParams['figure.figsize'] = [15, 8]


Nous allons travailler sur les valeurs mobilières `ada`.

Voici les points à vérifier quand les données sont chargées.

**Question(s)**:
1. charger vos données `ada` à partir du dictionnaire `Loader`
2. afficher le début et la fin de votre dataFrame. Quelles sont les dates de début et de fin de la série ?
3. quelle est la granularité temporelle utilisée ?
4. est ce que la série est stationnaire ?

In [ ]:
# Question 1
loader = ... # attention, mettre le paramètre dataDefaultPath='..'
dataInit = ...

In [ ]:
# Question 2

In [ ]:
# Question 3

In [ ]:
# Question 4

### De nouveaux indicateurs

Le rendement journalier (_daily return_) est la différence entre le prix à l'ouverture et le prix à la fermeture.

La volatilité journalière est la différence entre le prix le plus haut et le prix le plus bas.

**Question(s)**:
1. ajouter le rendement journalier dans une colonne `dailyReturn`
2. ajouter la volatilité dans une colonne `volatility`
3. afficher ces deux nouveaux indicateurs
4. est ce que ces 2 nouveaux indicateurs sont stationnaires ?

In [ ]:
# Questions 1-2


data = dataInit.copy()
...

In [ ]:
# Question 3


In [ ]:
# Question 4

### Passage en stationnaire

Nous allons utiliser les indicateurs suivant pour réaliser une prédiction à $t+1$ :
- le rendement journalier
- la volativité
- le volume journalier

La première phase est de rendre les données stationnaires : une méthode est de soustraire la _exponentially weighted moving average_ de votre indicateur et de diviser par la _exponentially weighted standard deviation_.

Nous avons discuté précédemment de _smoothing_, basé sur une moyenne glissante gauche ou centrée. La moyenne exponentiel permet de décroitre le poids des données éloignées, via une diminution de ce poid basé sur une fonction exponentielle (pour une moyenne gauche, donc en utilisant que les données à gauche de votre jour $t$, plus on s'éloigne de $t$, plus le poid associé à la donnée $t+i$ décroit vite).

**Question(s)**:
1. sélectionner les colonnes utiles dans un dataframe `dataSelec`
2. étudier la fonction `ewd` de `Pandas` (aide: cette fonction, commme `rolling` génére un "calcul fenêtré" - il est nécessaire d'appeler ensuite `mean()` ou `var()`, ou $\dots$ pour lancer une de ces fonctions sur les données fenêtrées)
3. créer un dataframe `ewdf`, à partir de `dataSelec`, contenant le _exponentially weighted moving average_ de vos indicateurs, avec le paramètre `halflife=10`
4. créer un dataframe `vewdf`, à partir de `dataSelec`, contenant le _exponentially weighted standard deviation_ de vos indicateurs, avec le paramètre `halflife=10`
5. pour chaque colonne `"Volume", "dailyReturn", "volatility"`, appliquer le traitement suivant :
   $$ dataReady[indicateur] = \frac{(dataSelec[indicateur] - ewdf[indicateur])}{\sqrt{vewdf[indicateur]}}$$

6. afficher vos indicateurs. Ont elles l'air plus stationnaires ?
7. réaliser le test de Augmented Dickey–Fuller (ADF), vu dans un des notebooks précédents


In [ ]:
# Questions 1-3

dataSelec = ...

ewdf = ...

ewdf.head()

In [ ]:
# Question 4

vewdf = ...

vewdf.head()

In [ ]:
# Question 5

dataReady = dataSelec.copy()

dataReady.volativity = ...
dataReady.Volume = ...
dataReady.dailyReturn = ...

dataReady = dataReady.dropna()

dataReady.head()

In [ ]:
# Question 6



In [ ]:
# Question 7


### Prédiction avec un réseau de neurones profond

Avant de parler de réseaux de neurones profonds, je vais d'abord vous présenter très brièvement ce qu'est un réseau de neurones artificiels. Ensuite, nous verrons brièvement les réseaux de neurones récurrents, adaptés pour gérer des séries temporelles.

#### Artificial Neural Network

Un réseau est composé de couches, chacune composée de noeuds. On distingue les couches d'entrée, les couches cachées et la couche de sortie :

<div>
<img src="../fig/ann.png" width="400"/>
</div>

Le nombre de couches cachées peut varier, le nombre de noeuds sur chaque couche peut aussi varier...il est donc nécessaire de fournir ces paramètres pour chaque réseau de neurones utilisé. L'architecture de votre réseau va dépendre du volume des vos données et de leur caractéristiques.

Voyons voir maintenant ce qu'est un noeud dans le réseau. Un noeud est une fonction prenant en paramètre une de vos données en entrée $(x, y, z)$ et retournant une valeur entre 0 et 1. 

$output_i = activation(ax + by + cz + d)$

où $activation()$ est une fonction d'activation de type sigmoïde (en forme de S). 

La sortie d'un noeud est ensuite l'entrée de la couche suivante. 

Voilà, vous savez tout. Pour que le réseau soit fonctionnel, il reste à "simplement" trouver les bons paramètres $(a,b,c)$, et ce pour chaque noeud...sur un réseau de 20 couches cachées, avec chacun 50 noeuds, cela fait $20 \cdot 50 \cdot 4 = 4000$ paramètres. 

En très bref, voici comment sont obtenus les paramètres :
1. initialisation aléatoire
2. feed forward : on donne les données en entrée au réseau et on calcule les sorties obtenues
3. on calcule l'erreur obtenu (ce qui implique que l'on sache ce que l'on doit obtenir pour chaque données en entrée)
4. backward propagation : on part de la sortie pour remettre à jour les paramètres de chaque noeud jusqu'à la première couche cachée. Les poids sont mis à jour par petit pas via la méthode de gradient (on se déplace par petit pas vers la solution optimale, en partant de notre initialisation)
5. revenir à l'étape 2, on s'arrête quand l'amélioration de l'erreur n'est plus significative.

Après ces explications, nous pouvons en déduire :
- il nous faut en entrée des données, beaucoup de données, pour apprendre les nombreux paramètres
- nous sommes dans un cas d'algorithme supervisé : les données en entrées sont étiquettées avec la valeur que l'on souhaite prédire. Nous connaissons les groupes de sortie ou bien la valeur à prédire.

#### Recurrent Neural Network

Rien de mieux que de citer Wikipédia pour commencer :

    Un réseau de neurones récurrents est un réseau de neurones artificiels présentant des connexions récurrentes. Un réseau de neurones récurrents est constitué d'unités (neurones) interconnectées interagissant non-linéairement et pour lequel il existe au moins un cycle dans la structure. Les unités sont reliées par des arcs (synapses) qui possèdent un poids. La sortie d'un neurone est une combinaison non linéaire de ses entrées.


Pour donner plus de sens à cette description, disons qu'un modèle récurrent permet de prendre en entrée des données ordonnées temporellement, et qu'un noeud du réseau de neurones au temps $t$ prend en entrée des données du noeud $t-1$. Cela permet de prendre en compte ce qu'il s'est passé à $t-1$ pour prédire la sortie au temps $t$. 

Pour tester ce modèle, nous allons utiliser une classe Strategy plus complexe que celle vu précédemment. Cette classe a toujours une méthode `getAction`, mais en a de nouvelles :
- `filterIn` et `filterOut` préparant les données d'apprentissages avant d'apprendre le modèle et réalisant l'opération inverse pour la prédiction obtenue. Les réseaux de neurones fonctionnenent parfois moieux quand les données sont normalisées
- `fit` permet d'apprendre les paramètres du modèle
- `predict` permet de récupérer une prédiction

Les paramètre à fournir à un modèle récurrent sont plus complexes que ceux de nos précédents modèles. Il faut tout d'abord définir la longueur de la série temporelle à prendre en entrée. Par exemple, sur une série `[12, 9, 8, 7, 15, 79, 23]`, on peut construire les données d'entrèes suivantes :
$$
\begin{bmatrix}
12 & 9 & 8\\
9 & 8 & 7\\
8 & 7 & 15\\
7 & 15 & 79\\
\end{bmatrix}
\begin{bmatrix}
7\\
15\\
79\\
23
\end{bmatrix}
$$

La matrice de gauche représente les séries temporelles d'entrées et le vecteur de droite les données que le modèle est censé prédire. Ce couple série temporelle / prédiction est utilisé pour apprendre les paramètres du modèle. Un paramètre à fournir est donc la taille des séries de la matrice de droite, que nous appellerons $n_steps$.

Un autre paramètre est le nombre de variables à utiliser pour nos données d'entrées. Dans l'exemple précédent, les séries temporelles ont une dimension de $1$ pour chaque élément (12, puis 9, $\dots$). Mais on peut aussi utiliser plusieurs variables d'entrées à chaque instant $t$. Par exemple, dans notre cas, nous allons utiliser le volume, le retour journalier et la volativité. Le paramètre `n_features` nous permettra de définir la dimension des variables à utiliser.

Ainsi nos données d'entrées sont définies sur trois dimensions $n \times n\_steps \times n\_features$, avec $n$ représentant le nombre d'instant (dans notre cas, le nombre de jour de la série de la valeur mobilière).

In [ ]:

n_steps = 31
n_features = 3


Un dernier paramètre à définir pour notre modèle est la date à laquelle apprendre le modèle initiale. En effet, pour notre stratégie, il est nécessaire de stocker tout d'abord des données, puis d'apprendre le modèle quand le volume de données stocké est suffisant. `startTrading` sert ici à définir la date à laquelle on commence à faire du trading.

Notre modèle `StrategyLSTM` prend donc en paramètre `startTrading, n_features, n_steps`. Le dernier paramètre indique quelle colonne utiliser comme données de prédiction. On indique ici que nous tenterons de prédire le `dailyReturn`. Nous passons ainsi seulement l'indice de la colonne du dataframe qui sera donné à la méthode `getAction` de notre classe `Strategy`.

In [ ]:
%run -i "../StrategiesModel.ipynb"

lastDate = dataInit.index[-1]
startTrading = date(2020, 1, 1) # year, month, day

strategyLSTM = StrategyLSTM(startTrading, n_features, n_steps, dataReady.columns.get_loc("dailyReturn"))

Nous pouvons maintenant exécuter notre jeu avec notre stratégie basée sur un réseau de neurones récurrent. On initialise le jeu :

In [ ]:

account = Account(100)

game = Game(dataInit.Close, data=dataReady)
game.setAccount(account)
game.setStrategy(strategyLSTM)



Et on execute le jeu :

In [ ]:
game.run(debug=True)

print(game)

Dans ce notebook, vous avez découvert de nouvelle variable à calculer sur vos données mobilières (volativité, retour journalier, ...) et un nouveau modèle, basé sur les réseaux de neurones pour obtenir une prédiction.

Vous pouvez maintenant consulter le code du fichier `StrategiesModel.ipynb` pour voir comment notre modèle est implémenté. L'idée maintenant est de tester d'autres paramètres, d'autres données, $\dots$